<a href="https://colab.research.google.com/github/MirmatlabKarimli/Thesis_homework/blob/main/notebooks/Lab_1_6_RAG_Extension_Thesis_AI_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q chromadb google-generativeai sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.5 MB/s eta

In [2]:
import os
from google.colab import userdata

GEMINI_KEY = userdata.get("GEMINI_KEY")
assert GEMINI_KEY is not None, "❌ Add GEMINI_KEY in Colab → Tools → Secrets."

print("Gemini key loaded successfully.")


Gemini key loaded successfully.


In [4]:
import google.generativeai as genai
genai.configure(api_key=GEMINI_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")


In [5]:
examples = []

import random

seasons = ["Winter", "Spring", "Summer", "Autumn"]

for i in range(1000):
    past_sales = [random.randint(20, 300) for _ in range(7)]
    season = random.choice(seasons)
    price = round(random.uniform(3, 15), 2)
    promotion = random.choice(["Yes", "No"])

    X = f"Past sales: {past_sales}, Season: {season}, Price: {price}, Promotion: {promotion}"

    trend = sum(past_sales[-3:]) / 3
    season_factor = {"Winter":1.2, "Spring":1.0, "Summer":0.9, "Autumn":1.1}[season]
    y_value = int(trend * season_factor)
    y = f"Predicted sales: {y_value}"

    examples.append({"X": X, "y": y})

len(examples)


1000

In [6]:
import chromadb
from sentence_transformers import SentenceTransformer

model_emb = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="drug_sales_examples")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
ids = []
docs = []
metas = []

for i, ex in enumerate(examples):
    ids.append(f"id_{i}")
    docs.append(ex["X"])
    metas.append({"y": ex["y"]})

embeddings = model_emb.encode(docs).tolist()

collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=docs,
    metadatas=metas
)

print("Inserted 1,000 examples into ChromaDB.")


Inserted 1,000 examples into ChromaDB.


In [8]:
def retrieve_similar_examples(query, top_k=3):
    query_emb = model_emb.encode([query]).tolist()[0]
    results = collection.query(query_embeddings=[query_emb], n_results=top_k)

    retrieved = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved.append((doc, meta["y"]))
    return retrieved


In [9]:
def build_rag_prompt(query, retrieved_examples):
    example_text = ""
    for x, y in retrieved_examples:
        example_text += f"Example:\nInput: {x}\nOutput: {y}\n\n"

    prompt = f"""
You are an AI agent for drug sales forecasting.

Here are retrieved examples similar to the new input:

{example_text}

Now process the new input:
{query}

Explain your reasoning and provide the final predicted sales.
"""
    return prompt


In [10]:
input_query = """
Past sales: [140, 160, 180, 200, 210, 240, 260],
Season: Winter,
Price: 6.5,
Promotion: No
"""

retrieved = retrieve_similar_examples(input_query)
retrieved


[('Past sales: [206, 140, 228, 36, 82, 29, 86], Season: Winter, Price: 12.58, Promotion: No',
  'Predicted sales: 78'),
 ('Past sales: [211, 140, 132, 65, 150, 176, 180], Season: Winter, Price: 12.15, Promotion: Yes',
  'Predicted sales: 202'),
 ('Past sales: [106, 200, 171, 140, 58, 45, 195], Season: Winter, Price: 12.41, Promotion: No',
  'Predicted sales: 119')]

In [11]:
rag_prompt = build_rag_prompt(input_query, retrieved)
response = model.generate_content(rag_prompt)

print(response.text)


Reasoning:

1.  **Analyze Past Sales Trend:** The past sales show a strong and consistent upward trend: [140, 160, 180, 200, 210, 240, 260]. The most recent sales are significantly higher than the earlier ones. This indicates growing demand and momentum, suggesting that future sales will likely be higher than the overall average (which is ~198.57) and potentially higher than the last observed sale (260). The average of the last three sales (210, 240, 260) is ~236.67.

2.  **Evaluate Price:** The price is 6.5. This is *drastically lower* than the prices in the provided examples (12.58, 12.15, 12.41). A significantly lower price typically acts as a powerful driver for increased sales volume, making the product much more attractive and accessible to a wider market. This is a very strong positive factor.

3.  **Consider Promotion:** The promotion status is "No." In Example 1 and Example 3, where promotion was "No" and prices were high (~12.xx), the predicted sales were lower than the avera

In [12]:
print("""
RAG Pipeline
-------------------------
Input X
  ↓
Vector Search (Top 3)
  ↓
Augmented Prompt
  ↓
Gemini Reasoning + Output (Y)
-------------------------
""")



RAG Pipeline
-------------------------
Input X
  ↓
Vector Search (Top 3)
  ↓
Augmented Prompt
  ↓
Gemini Reasoning + Output (Y)
-------------------------



✅ Reflection

Integrating RAG significantly improved the performance and stability of my thesis-based AI system. By storing 1,000 synthetic drug sales forecasting examples in a vector database, the model can now retrieve highly relevant patterns before generating predictions. This retrieval step provides Gemini with domain-specific context, leading to outputs that are more accurate, more consistent, and better aligned with realistic pharmaceutical sales behavior. Without RAG, the model relies purely on prompt reasoning; with RAG, it benefits from concrete examples that guide its inference process. The top-3 similarity retrieval helps the model generalize better, especially in scenarios involving seasonal fluctuations or noisy historical sales data. The system also became more explainable, as retrieved examples show why the model predicts a certain trend. A limitation is that the examples are synthetic; incorporating real pharmacy data would further improve reliability. Nonetheless, this lab demonstrates that RAG transforms the AI agent from a purely generative system into a hybrid reasoning engine grounded in actual data patterns.